#### Setup & Imports

In [47]:
# =============================================================================
# Cell 1: Setup & Imports
# =============================================================================
"""
02_SSD_EfficientDet_Comparison.ipynb
------------------------------------
Train SSDLite and EfficientDet for hand detection using the same dataset
as the YOLO notebook for fair comparison.
Models (all ~2-4M parameters):
- SSDLite320-MobileNetV3-Large (~2.2M params)
- EfficientDet-D0 (~3.9M params)
- YOLOv8n (~3.2M params) - already trained
Dataset: Same YOLO dataset (EgoHands + HaGRID + Oxford)
"""
import os
import sys
import time
import json
import random
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from torchvision.models.detection import (
    ssdlite320_mobilenet_v3_large,
    SSDLite320_MobileNet_V3_Large_Weights,
)
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
# Check for effdet
try:
    import effdet
    EFFDET_AVAILABLE = True
    print("effdet is installed")
except ImportError:
    EFFDET_AVAILABLE = False
    print("WARNING: effdet not installed!")
    print("Please run: pip install effdet")
    print("Then restart the kernel.")
# Check environment
print("\n" + "="*60)
print("ENVIRONMENT SETUP")
print("="*60)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Random seed: {SEED}")

effdet is installed

ENVIRONMENT SETUP
PyTorch version: 2.9.1+cu128
Torchvision version: 0.24.1+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA version: 12.8
Random seed: 42


#### Path Configuration

In [48]:
# =============================================================================
# Cell 2: Path Configuration
# =============================================================================
"""
We'll use the same dataset created in the YOLO notebook.
The YOLO format will be converted to COCO format for torchvision models.
"""
# Base paths - adjust if needed
NOTEBOOK_DIR = Path.cwd()
PROJ_ROOT = NOTEBOOK_DIR / "hand_bbox_baseline"
# Dataset paths (from YOLO notebook)
YOLO_DATASET_DIR = PROJ_ROOT / "data" / "hands_yolo"
DATA_YAML_PATH = YOLO_DATASET_DIR / "data.yaml"
# Output directory for this notebook
RUNS_DIR = PROJ_ROOT / "runs_torchvision"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
# Verify dataset exists
print("="*60)
print("PATH CONFIGURATION")
print("="*60)
print(f"Project root: {PROJ_ROOT}")
print(f"YOLO dataset: {YOLO_DATASET_DIR}")
print(f"Output runs: {RUNS_DIR}")
# Check dataset
if not YOLO_DATASET_DIR.exists():
    raise FileNotFoundError(f"YOLO dataset not found at {YOLO_DATASET_DIR}. Run YOLO notebook first!")
for split in ["train", "val", "test"]:
    img_dir = YOLO_DATASET_DIR / "images" / split
    lbl_dir = YOLO_DATASET_DIR / "labels" / split
    n_imgs = len(list(img_dir.glob("*.[jJ][pP][gG]"))) + len(list(img_dir.glob("*.[pP][nN][gG]")))
    n_lbls = len(list(lbl_dir.glob("*.txt")))
    print(f"  {split}: {n_imgs} images, {n_lbls} labels")
print("\nDataset verified!")

PATH CONFIGURATION
Project root: /home/fransando/Dev/tfg-sign-language-recognition/01_notebooks/hand_detection/hand_bbox_baseline
YOLO dataset: /home/fransando/Dev/tfg-sign-language-recognition/01_notebooks/hand_detection/hand_bbox_baseline/data/hands_yolo
Output runs: /home/fransando/Dev/tfg-sign-language-recognition/01_notebooks/hand_detection/hand_bbox_baseline/runs_torchvision
  train: 10577 images, 10577 labels
  val: 1321 images, 1321 labels
  test: 1325 images, 1325 labels

Dataset verified!


#### Training Configuration

In [49]:
# =============================================================================
# Cell 3: Training Configuration (Same as YOLO for fair comparison)
# =============================================================================
# Training config
CONFIG = {
    # Dataset
    "img_size_ssd": 320,       # SSDLite320 native size
    "img_size_effdet": 512,    # EfficientDet-D0 native size
    
    # Training
    "batch_size": 16,
    "epochs": 100,
    "patience": 25,
    
    # Optimizer (same as YOLO)
    "optimizer": "AdamW",
    "lr": 0.001,
    "weight_decay": 0.0005,
    "momentum": 0.9,
    
    # LR Scheduler
    "lr_scheduler": "cosine",
    "warmup_epochs": 3,
    
    # Other
    "num_workers": 4,
    "pin_memory": True,
}
# Models to train - all ~2-4M parameters for fair comparison
MODELS_TO_TRAIN = [
    {
        "name": "ssdlite320_mobilenetv3",
        "model_type": "ssd",
        "img_size": CONFIG["img_size_ssd"],
        "batch_size": 16,  # SSD can handle 16
    },
    {
        "name": "efficientdet_d0",
        "model_type": "effdet",
        "img_size": CONFIG["img_size_effdet"],
        "batch_size": 4,  # Reduce to 4 for 512x512 images
    },
]
print("="*60)
print("TRAINING CONFIGURATION")
print("="*60)
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print("\nModels to train:")
for m in MODELS_TO_TRAIN:
    print(f"  - {m['name']} (img_size={m['img_size']})")

TRAINING CONFIGURATION
  img_size_ssd: 320
  img_size_effdet: 512
  batch_size: 16
  epochs: 100
  patience: 25
  optimizer: AdamW
  lr: 0.001
  weight_decay: 0.0005
  momentum: 0.9
  lr_scheduler: cosine
  warmup_epochs: 3
  num_workers: 4
  pin_memory: True

Models to train:
  - ssdlite320_mobilenetv3 (img_size=320)
  - efficientdet_d0 (img_size=512)


#### Dataset Class

In [50]:
# =============================================================================
# Cell 4: Dataset Class (YOLO format -> torchvision format) - FIXED
# =============================================================================
"""
Torchvision detection models expect:
- image: tensor [C, H, W]
- target: dict with:
    - boxes: [N, 4] in (x1, y1, x2, y2) pixel coordinates
    - labels: [N] class labels (1-indexed, 0 is background)
"""
class HandDetectionDataset(Dataset):
    """
    Load YOLO format dataset for torchvision detection models.
    """
    def __init__(self, yolo_root: Path, split: str, img_size: int = 320, transforms=None):
        self.img_dir = yolo_root / "images" / split
        self.lbl_dir = yolo_root / "labels" / split
        self.img_size = img_size
        self.transforms = transforms
        
        # Collect all images
        self.img_paths = sorted(
            list(self.img_dir.glob("*.jpg")) + 
            list(self.img_dir.glob("*.png")) +
            list(self.img_dir.glob("*.jpeg"))
        )
        
        print(f"  {split}: {len(self.img_paths)} images")
    
    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        lbl_path = self.lbl_dir / (img_path.stem + ".txt")
        
        # Load image
        img = Image.open(img_path).convert("RGB")
        orig_w, orig_h = img.size
        
        # Resize image to exact size required
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        
        # Load YOLO labels and convert to pixel coords
        boxes = []
        labels = []
        
        if lbl_path.exists():
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    
                    cls_id, cx, cy, w, h = map(float, parts)
                    
                    # Convert YOLO normalized (cx, cy, w, h) to pixel (x1, y1, x2, y2)
                    cx_px = cx * self.img_size
                    cy_px = cy * self.img_size
                    w_px = w * self.img_size
                    h_px = h * self.img_size
                    
                    x1 = cx_px - w_px / 2
                    y1 = cy_px - h_px / 2
                    x2 = cx_px + w_px / 2
                    y2 = cy_px + h_px / 2
                    
                    # Clamp to image bounds
                    x1 = max(0, min(self.img_size - 1, x1))
                    y1 = max(0, min(self.img_size - 1, y1))
                    x2 = max(x1 + 1, min(self.img_size, x2))  # Ensure x2 > x1
                    y2 = max(y1 + 1, min(self.img_size, y2))  # Ensure y2 > y1
                    
                    # Skip degenerate boxes (too small)
                    if (x2 - x1) < 2 or (y2 - y1) < 2:
                        continue
                    
                    boxes.append([x1, y1, x2, y2])
                    labels.append(1)  # Class 1 = hand (0 is background in torchvision)
        
        # Convert to tensors
        if len(boxes) > 0:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        else:
            # Empty annotations - create dummy box (will be filtered during training)
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        
        # Image to tensor
        img_tensor = transforms.ToTensor()(img)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
        }
        
        return img_tensor, target
def collate_fn(batch):
    """Custom collate that filters out samples with no boxes."""
    # Filter out samples with empty boxes
    batch = [(img, tgt) for img, tgt in batch if len(tgt["boxes"]) > 0]
    
    if len(batch) == 0:
        # Return a minimal valid batch if all samples were filtered
        return None
    
    return tuple(zip(*batch))
# Test dataset loading
print("="*60)
print("DATASET LOADING TEST")
print("="*60)
test_dataset = HandDetectionDataset(YOLO_DATASET_DIR, "train", img_size=320)
img, target = test_dataset[0]
print(f"Image shape: {img.shape}")
print(f"Boxes shape: {target['boxes'].shape}")
print(f"Labels: {target['labels']}")
print(f"Image size matches expected: {img.shape[1]} x {img.shape[2]} = 320 x 320")

DATASET LOADING TEST
  train: 10577 images
Image shape: torch.Size([3, 320, 320])
Boxes shape: torch.Size([4, 4])
Labels: tensor([1, 1, 1, 1])
Image size matches expected: 320 x 320 = 320 x 320


#### Model Factory

In [51]:
# =============================================================================
# Cell 5: Model Factory (Create SSD and EfficientDet for 1 class)
# =============================================================================
"""
Lightweight models for fair comparison with YOLOv8n (~3.2M params).
Models:
- SSDLite320-MobileNetV3-Large: ~2.2M params (single-shot, anchor-based)
- EfficientDet-D0: ~3.9M params (single-shot, BiFPN architecture)
Both are single-shot detectors but with different architectures:
- SSD: Simple feature pyramid
- EfficientDet: Bidirectional Feature Pyramid Network (BiFPN)
"""
from functools import partial
# Install effdet if needed (run once):
# !pip install effdet
try:
    from effdet import get_efficientdet_config, EfficientDet, DetBenchTrain, DetBenchPredict
    from effdet.efficientdet import HeadNet
    EFFDET_AVAILABLE = True
except ImportError:
    EFFDET_AVAILABLE = False
    print("WARNING: effdet not installed. Run: pip install effdet")
def create_ssdlite_model(num_classes: int = 2, pretrained: bool = True):
    """
    Create SSDLite320-MobileNetV3-Large for custom number of classes.
    ~2.2M parameters
    """
    if pretrained:
        weights = SSDLite320_MobileNet_V3_Large_Weights.COCO_V1
        model = ssdlite320_mobilenet_v3_large(weights=weights)
    else:
        model = ssdlite320_mobilenet_v3_large(weights=None, num_classes=num_classes)
        return model
    
    # Replace classification head
    in_channels = [672, 480, 512, 256, 256, 128]
    num_anchors = model.anchor_generator.num_anchors_per_location()
    
    norm_layer = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
    
    model.head.classification_head = SSDLiteClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
        norm_layer=norm_layer,
    )
    
    return model
def create_efficientdet_model(num_classes: int = 1, pretrained: bool = True, img_size: int = 512):
    """
    Create EfficientDet-D0 for custom number of classes.
    ~3.9M parameters
    
    Note: EfficientDet uses num_classes WITHOUT background class (unlike torchvision)
    So for hand detection: num_classes=1 (just "hand")
    """
    if not EFFDET_AVAILABLE:
        raise ImportError("effdet not installed. Run: pip install effdet")
    
    # Get config for EfficientDet-D0
    config = get_efficientdet_config('tf_efficientdet_d0')
    config.num_classes = num_classes
    config.image_size = (img_size, img_size)
    
    # Create model
    net = EfficientDet(config, pretrained_backbone=pretrained)
    
    # Replace the class/box head for our number of classes
    net.class_net = HeadNet(
        config,
        num_outputs=config.num_classes,
    )
    
    # Load pretrained weights if available (COCO)
    if pretrained:
        try:
            from effdet.efficientdet import load_pretrained
            load_pretrained(net, config.url)
            # Re-initialize the class head since we changed num_classes
            net.class_net = HeadNet(config, num_outputs=config.num_classes)
        except Exception as e:
            print(f"Could not load pretrained weights: {e}")
            print("Using ImageNet pretrained backbone only.")
    
    return net, config
class EfficientDetWrapper(nn.Module):
    """
    Wrapper to make EfficientDet compatible with torchvision-style training.
    
    Training: forward(images, targets) -> loss_dict
    Inference: forward(images) -> list of {boxes, scores, labels}
    """
    def __init__(self, num_classes: int = 1, pretrained: bool = True, img_size: int = 512):
        super().__init__()
        
        if not EFFDET_AVAILABLE:
            raise ImportError("effdet not installed. Run: pip install effdet")
        
        # Get config
        self.config = get_efficientdet_config('tf_efficientdet_d0')
        self.config.num_classes = num_classes
        self.config.image_size = (img_size, img_size)
        
        # Create model for training
        net = EfficientDet(self.config, pretrained_backbone=pretrained)
        
        # Load COCO pretrained weights for the backbone
        if pretrained:
            try:
                from effdet import create_model
                pretrained_model = create_model(
                    'tf_efficientdet_d0',
                    bench_task='train',
                    num_classes=90,  # COCO classes
                    pretrained=True,
                )
                # Copy backbone weights
                net.backbone.load_state_dict(pretrained_model.model.backbone.state_dict())
                del pretrained_model
            except Exception as e:
                print(f"Note: Using ImageNet backbone only. ({e})")
        
        # Wrap for training (handles loss computation)
        self.train_model = DetBenchTrain(net, self.config)
        
        # Keep reference to base model for inference
        self.model = net
        
        self.img_size = img_size
        self.num_classes = num_classes
    
    def forward(self, images, targets=None):
        """
        Args:
            images: list of tensors [C, H, W] or batched tensor [B, C, H, W]
            targets: list of dicts with 'boxes' and 'labels' (training only)
        
        Returns:
            Training: dict with losses
            Inference: list of dicts with 'boxes', 'scores', 'labels'
        """
        # Stack images if list
        if isinstance(images, (list, tuple)):
            images = torch.stack(images)
        
        if self.training and targets is not None:
            # Training mode - compute losses
            # Convert targets to effdet format
            batch_size = images.shape[0]
            
            # effdet expects targets as dict with 'bbox' and 'cls'
            # bbox format: [x1, y1, x2, y2] (absolute coordinates)
            # cls: class indices (0-indexed, no background)
            
            target_boxes = []
            target_cls = []
            
            for t in targets:
                boxes = t['boxes']  # [N, 4]
                labels = t['labels'] - 1  # Convert from 1-indexed to 0-indexed
                
                # Pad to same size
                target_boxes.append(boxes)
                target_cls.append(labels)
            
            # Pad targets to same length
            max_len = max(len(b) for b in target_boxes) if target_boxes else 0
            if max_len == 0:
                max_len = 1
            
            padded_boxes = torch.zeros(batch_size, max_len, 4, device=images.device)
            padded_cls = torch.zeros(batch_size, max_len, device=images.device, dtype=torch.long)
            
            for i, (boxes, cls) in enumerate(zip(target_boxes, target_cls)):
                if len(boxes) > 0:
                    padded_boxes[i, :len(boxes)] = boxes
                    padded_cls[i, :len(cls)] = cls
            
            target_dict = {
                'bbox': padded_boxes,
                'cls': padded_cls,
            }
            
            # Forward through training bench
            loss_dict = self.train_model(images, target_dict)
            
            return loss_dict
        
        else:
            # Inference mode
            self.model.eval()
            
            with torch.no_grad():
                # Use DetBenchPredict for inference
                bench = DetBenchPredict(self.model, self.config)
                bench.eval()
                bench.to(images.device)
                
                # Get detections
                detections = bench(images)
                
                # Convert to torchvision format
                # detections shape: [batch, num_detections, 6]
                # Each detection: [x1, y1, x2, y2, score, class_id]
                
                results = []
                for det in detections:
                    # Filter valid detections (score > 0)
                    valid = det[:, 4] > 0.01
                    det = det[valid]
                    
                    results.append({
                        'boxes': det[:, :4],
                        'scores': det[:, 4],
                        'labels': det[:, 5].long() + 1,  # Convert back to 1-indexed
                    })
                
                return results
def count_parameters(model):
    """Count model parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable
# Test model creation
print("="*60)
print("MODEL CREATION TEST")
print("="*60)
model_params = {}
# Test SSDLite
print("\nSSDLite320-MobileNetV3:")
model = create_ssdlite_model(num_classes=2)
total, trainable = count_parameters(model)
model_params['SSDLite320-MobileNetV3'] = total / 1e6
print(f"  Total params: {total:,} ({total/1e6:.2f}M)")
print(f"  Trainable params: {trainable:,} ({trainable/1e6:.2f}M)")
del model
# Test EfficientDet
if EFFDET_AVAILABLE:
    print("\nEfficientDet-D0:")
    model = EfficientDetWrapper(num_classes=1, pretrained=True, img_size=512)
    total, trainable = count_parameters(model)
    model_params['EfficientDet-D0'] = total / 1e6
    print(f"  Total params: {total:,} ({total/1e6:.2f}M)")
    print(f"  Trainable params: {trainable:,} ({trainable/1e6:.2f}M)")
    del model
else:
    print("\nEfficientDet-D0: NOT AVAILABLE (install effdet)")
    model_params['EfficientDet-D0'] = 3.9  # Expected value
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("\n" + "="*60)
print("PARAMETER COMPARISON")
print("="*60)
print(f"  YOLOv8n:                ~3.20M (reference)")
print(f"  SSDLite320-MobileNetV3: ~{model_params.get('SSDLite320-MobileNetV3', 0):.2f}M")
print(f"  EfficientDet-D0:        ~{model_params.get('EfficientDet-D0', 0):.2f}M")
print("="*60)
# Check if comparison is fair
all_params = [3.2, model_params.get('SSDLite320-MobileNetV3', 0), 
              model_params.get('EfficientDet-D0', 3.9)]
max_param = max(all_params)
min_param = min(all_params)
ratio = max_param / min_param
if ratio < 2:
    print(f"\nExcellent comparison: All models within {ratio:.1f}x parameter range")
elif ratio < 3:
    print(f"\nFair comparison: All models within {ratio:.1f}x parameter range")
else:
    print(f"\nWARNING: Parameter ratio is {ratio:.1f}x - may not be fair")
print("\nModels created successfully!")

MODEL CREATION TEST

SSDLite320-MobileNetV3:
  Total params: 2,206,520 (2.21M)
  Trainable params: 2,206,520 (2.21M)

EfficientDet-D0:


  Total params: 3,828,002 (3.83M)
  Trainable params: 3,828,002 (3.83M)

PARAMETER COMPARISON
  YOLOv8n:                ~3.20M (reference)
  SSDLite320-MobileNetV3: ~2.21M
  EfficientDet-D0:        ~3.83M

Excellent comparison: All models within 1.7x parameter range

Models created successfully!


#### Training Utilities

In [52]:
# =============================================================================
# Cell 6: Training Utilities
# =============================================================================
class EarlyStopping:
    """Early stopping to prevent overfitting."""
    def __init__(self, patience: int = 10, min_delta: float = 0.0, mode: str = "max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        
    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
        
        if self.mode == "max":
            improved = score > self.best_score + self.min_delta
        else:
            improved = score < self.best_score - self.min_delta
        
        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        
        return self.early_stop
class AverageMeter:
    """Compute and store the average and current value."""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count
def get_lr_scheduler(optimizer, config, steps_per_epoch):
    """Create learning rate scheduler with warmup."""
    warmup_steps = config["warmup_epochs"] * steps_per_epoch
    total_steps = config["epochs"] * steps_per_epoch
    
    def lr_lambda(step):
        if step < warmup_steps:
            # Linear warmup
            return step / warmup_steps
        else:
            # Cosine decay
            progress = (step - warmup_steps) / (total_steps - warmup_steps)
            return 0.5 * (1 + np.cos(np.pi * progress))
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print("Training utilities loaded!")

Training utilities loaded!


#### Evaluation Functions

In [53]:
# =============================================================================
# Cell 7: Evaluation Functions (FIXED - handles edge cases)
# =============================================================================
def compute_iou(box1, box2):
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0
def compute_ap(precisions, recalls):
    """Compute Average Precision using 11-point interpolation."""
    if not precisions or not recalls:
        return 0.0
    ap = 0
    for t in np.linspace(0, 1, 11):
        precisions_above_t = [p for p, r in zip(precisions, recalls) if r >= t]
        if precisions_above_t:
            ap += max(precisions_above_t) / 11
    return ap
@torch.no_grad()
def evaluate_model(model, dataloader, device, conf_threshold=0.5, iou_threshold=0.5, model_type="ssd"):
    """
    Evaluate detection model and compute metrics.
    Returns: dict with mAP50, mAP50-95, precision, recall, and IoU stats.
    """
    model.eval()
    
    all_predictions = []
    total_gt = 0
    all_ious = []
    inference_times = []
    
    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        # Skip None batches
        if batch is None:
            continue
            
        images, targets = batch
        
        # Skip empty batches
        if len(images) == 0:
            continue
        
        images_device = [img.to(device) for img in images]
        
        # Inference with timing
        start_time = time.time()
        
        try:
            if model_type == "effdet":
                images_stacked = torch.stack(images_device)
                outputs = model(images_stacked)
            else:
                outputs = model(images_device)
        except Exception as e:
            print(f"  Warning: Inference error, skipping batch: {e}")
            continue
        
        inference_times.append(time.time() - start_time)
        
        # Process each image
        for output, target in zip(outputs, targets):
            pred_boxes = output["boxes"].cpu().numpy()
            pred_scores = output["scores"].cpu().numpy()
            pred_labels = output["labels"].cpu().numpy()
            
            gt_boxes = target["boxes"].numpy()
            total_gt += len(gt_boxes)
            
            # Filter by confidence and class
            mask = (pred_scores >= conf_threshold) & (pred_labels == 1)
            pred_boxes = pred_boxes[mask]
            pred_scores = pred_scores[mask]
            
            # Match predictions to ground truth
            gt_matched = [False] * len(gt_boxes)
            sorted_indices = np.argsort(-pred_scores)
            
            for idx in sorted_indices:
                pred_box = pred_boxes[idx]
                pred_conf = pred_scores[idx]
                
                best_iou = 0
                best_gt_idx = -1
                
                for gt_idx, gt_box in enumerate(gt_boxes):
                    if gt_matched[gt_idx]:
                        continue
                    iou = compute_iou(pred_box, gt_box)
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = gt_idx
                
                if best_iou >= iou_threshold and best_gt_idx >= 0:
                    gt_matched[best_gt_idx] = True
                    all_predictions.append((pred_conf, True, best_iou))
                    all_ious.append(best_iou)
                else:
                    all_predictions.append((pred_conf, False, best_iou))
    
    # Compute metrics
    if not all_predictions:
        return {
            "mAP50": 0.0,
            "mAP50-95": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "iou_stats": {},
            "speed_stats": {"mean_inference_ms": 0, "fps": 0},
            "total_predictions": 0,
            "total_gt": total_gt,
        }
    
    # Sort by confidence
    all_predictions.sort(key=lambda x: -x[0])
    
    # Compute precision-recall
    tp = 0
    fp = 0
    precisions = []
    recalls = []
    
    for conf, is_tp, iou in all_predictions:
        if is_tp:
            tp += 1
        else:
            fp += 1
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / total_gt if total_gt > 0 else 0
        precisions.append(precision)
        recalls.append(recall)
    
    ap50 = compute_ap(precisions, recalls)
    
    # Compute mAP50-95
    aps = []
    for iou_thresh in np.arange(0.5, 1.0, 0.05):
        tp_at_thresh = 0
        precs = []
        recs = []
        for i, (conf, is_tp, iou) in enumerate(all_predictions):
            if is_tp and iou >= iou_thresh:
                tp_at_thresh += 1
            precision = tp_at_thresh / (i + 1)
            recall = tp_at_thresh / total_gt if total_gt > 0 else 0
            precs.append(precision)
            recs.append(recall)
        aps.append(compute_ap(precs, recs))
    
    map50_95 = np.mean(aps) if aps else 0
    
    # Final metrics
    final_precision = precisions[-1] if precisions else 0
    final_recall = recalls[-1] if recalls else 0
    
    # IoU stats
    iou_stats = {}
    if all_ious:
        all_ious = np.array(all_ious)
        iou_stats = {
            "mean_iou": float(np.mean(all_ious)),
            "median_iou": float(np.median(all_ious)),
            "std_iou": float(np.std(all_ious)),
        }
    
    # Speed stats
    speed_stats = {"mean_inference_ms": 0, "fps": 0}
    if inference_times:
        inference_times = np.array(inference_times)
        speed_stats = {
            "mean_inference_ms": float(np.mean(inference_times) * 1000),
            "fps": float(len(images) / np.mean(inference_times)) if np.mean(inference_times) > 0 else 0,
        }
    
    return {
        "mAP50": ap50,
        "mAP50-95": map50_95,
        "precision": final_precision,
        "recall": final_recall,
        "iou_stats": iou_stats,
        "speed_stats": speed_stats,
        "total_predictions": len(all_predictions),
        "total_gt": total_gt,
    }
print("Evaluation functions loaded!")

Evaluation functions loaded!


#### Training Loop

In [54]:
# =============================================================================
# Cell 8: Training Loop (FIXED - handles batch size issues)
# =============================================================================
def train_one_epoch(model, dataloader, optimizer, scheduler, device, epoch, model_type="ssd"):
    """Train for one epoch."""
    model.train()

    loss_meter = AverageMeter()

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=False)

    for batch in pbar:
        # Skip None batches (all samples filtered out)
        if batch is None:
            continue

        images, targets = batch

        # Skip batches with only 1 sample (BatchNorm issue)
        if len(images) < 2:
            continue

        # Move to device
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        try:
            # Forward pass - handle different model types
            if model_type == "effdet":
                # EfficientDet expects stacked images
                images_stacked = torch.stack(images)
                loss_dict = model(images_stacked, targets)

                # EfficientDet returns {'loss': total_loss} or similar
                if isinstance(loss_dict, dict):
                    losses = sum(loss for loss in loss_dict.values() if isinstance(loss, torch.Tensor))
                else:
                    losses = loss_dict
            else:
                # SSD (torchvision style)
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())

            # Check for NaN loss
            if torch.isnan(losses):
                print(f"  Warning: NaN loss detected, skipping batch")
                continue

            # Backward pass
            optimizer.zero_grad()
            losses.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)

            optimizer.step()
            scheduler.step()

            # Update metrics
            loss_meter.update(losses.item(), len(images))

            pbar.set_postfix({
                "loss": f"{loss_meter.avg:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.6f}"
            })

        except Exception as e:
            print(f"  Warning: Error in batch, skipping: {e}")
            continue

    return loss_meter.avg


def train_model(model_cfg, config, yolo_dataset_dir, runs_dir, device):
    """Full training pipeline for a single model."""

    name = model_cfg["name"]
    model_type = model_cfg["model_type"]
    img_size = model_cfg["img_size"]

    # Use model-specific batch size if available, otherwise use config default
    batch_size = model_cfg.get("batch_size", config["batch_size"])

    print("\n" + "="*70)
    print(f"TRAINING: {name}")
    print("="*70)

    # Create output directory
    run_dir = runs_dir / name
    run_dir.mkdir(parents=True, exist_ok=True)
    weights_dir = run_dir / "weights"
    weights_dir.mkdir(exist_ok=True)

    # Create datasets
    print("\nLoading datasets...")
    train_dataset = HandDetectionDataset(yolo_dataset_dir, "train", img_size=img_size)
    val_dataset = HandDetectionDataset(yolo_dataset_dir, "val", img_size=img_size)

    # Use drop_last=True to avoid batch size 1 issues
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=config["num_workers"],
        pin_memory=config["pin_memory"],
        collate_fn=collate_fn,
        drop_last=True,  # DROP LAST INCOMPLETE BATCH
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=config["pin_memory"],
        collate_fn=collate_fn,
        drop_last=False,  # Keep all validation samples
    )

    # Create model
    print("\nCreating model...")
    if model_type == "ssd":
        model = create_ssdlite_model(num_classes=2)
    elif model_type == "effdet":
        model = EfficientDetWrapper(num_classes=1, pretrained=True, img_size=img_size)
    else:
        raise ValueError(f"Unknown model type: {model_type}")

    model.to(device)

    total_params, _ = count_parameters(model)
    print(f"Model parameters: {total_params/1e6:.2f}M")

    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    # Scheduler
    scheduler = get_lr_scheduler(optimizer, config, len(train_loader))

    # Early stopping
    early_stopping = EarlyStopping(patience=config["patience"], mode="max")

    # Training history
    history = {
        "train_loss": [],
        "val_mAP50": [],
        "val_mAP50-95": [],
        "lr": [],
    }

    best_map50 = 0

    print(f"\nStarting training for {config['epochs']} epochs...")
    print(f"  Batch size: {batch_size}")
    print(f"  Learning rate: {config['lr']}")
    print(f"  Early stopping patience: {config['patience']}")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")

    start_time = time.time()

    for epoch in range(config["epochs"]):
        # Train
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scheduler, device, epoch,
            model_type=model_type
        )

        # Validate
        val_metrics = evaluate_model(
            model, val_loader, device,
            conf_threshold=0.25,
            model_type=model_type
        )

        # Log
        current_lr = scheduler.get_last_lr()[0]
        history["train_loss"].append(train_loss)
        history["val_mAP50"].append(val_metrics["mAP50"])
        history["val_mAP50-95"].append(val_metrics["mAP50-95"])
        history["lr"].append(current_lr)

        print(f"Epoch {epoch+1}/{config['epochs']}: "
            f"loss={train_loss:.4f}, "
            f"mAP50={val_metrics['mAP50']:.4f}, "
            f"mAP50-95={val_metrics['mAP50-95']:.4f}, "
            f"lr={current_lr:.6f}")

        # Save best model
        if val_metrics["mAP50"] > best_map50:
            best_map50 = val_metrics["mAP50"]
            torch.save(model.state_dict(), weights_dir / "best.pt")
            print(f"  -> New best mAP50: {best_map50:.4f}")

        # Save last model
        torch.save(model.state_dict(), weights_dir / "last.pt")

        # Early stopping
        if early_stopping(val_metrics["mAP50"]):
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

    train_time = time.time() - start_time
    print(f"\nTraining completed in {train_time/60:.1f} minutes")

    # Save history
    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "history.csv", index=False)

    # Save config
    with open(run_dir / "config.json", "w") as f:
        json.dump({**config, **model_cfg, "total_params_M": total_params/1e6}, f, indent=2)

    return {
        "name": name,
        "run_dir": run_dir,
        "best_map50": best_map50,
        "train_time_min": train_time / 60,
        "total_epochs": len(history["train_loss"]),
    }


print("Training loop loaded!")

Training loop loaded!


#### Training

In [55]:
# =============================================================================
# Cell 9: Run Training for All Models
# =============================================================================
"""
Train both SSDLite and Faster R-CNN models.
This will take several hours depending on your GPU.
"""
trained_models = []
for model_cfg in [MODELS_TO_TRAIN[1]]:
    result = train_model(
        model_cfg=model_cfg,
        config=CONFIG,
        yolo_dataset_dir=YOLO_DATASET_DIR,
        runs_dir=RUNS_DIR,
        device=device,
    )
    trained_models.append(result)
    
    # Clear GPU memory between models
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
# Summary
print("\n" + "="*70)
print("TRAINING SUMMARY")
print("="*70)
for result in trained_models:
    print(f"\n{result['name']}:")
    print(f"  Best mAP50: {result['best_map50']:.4f}")
    print(f"  Epochs: {result['total_epochs']}")
    print(f"  Time: {result['train_time_min']:.1f} min")
    print(f"  Weights: {result['run_dir'] / 'weights' / 'best.pt'}")


TRAINING: efficientdet_d0

Loading datasets...
  train: 10577 images
  val: 1321 images

Creating model...


Model parameters: 3.83M

Starting training for 100 epochs...
  Batch size: 4
  Learning rate: 0.001
  Early stopping patience: 25
  Train batches: 2644
  Val batches: 331


KeyboardInterrupt: 

#### Final Evalutaion on Test Set

In [ ]:
# =============================================================================
# Cell 10: Final Evaluation on Test Set
# =============================================================================
"""
Evaluate all trained models on the test set with comprehensive metrics.
"""
def evaluate_final(model_name, weights_path, img_size, device):
    """Load model and evaluate on test set."""
    
    print(f"\nEvaluating {model_name}...")
    
    # Load model
    if "ssd" in model_name:
        model = create_ssdlite_model(num_classes=2)
    else:
        model = create_fasterrcnn_model(num_classes=2)
    
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()
    
    # Create test dataloader
    test_dataset = HandDetectionDataset(YOLO_DATASET_DIR, "test", img_size=img_size)
    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        collate_fn=collate_fn,
    )
    
    # Evaluate
    metrics = evaluate_model(model, test_loader, device, conf_threshold=0.25)
    
    # Add model info
    total_params, _ = count_parameters(model)
    metrics["parameters_M"] = total_params / 1e6
    metrics["model_name"] = model_name
    
    return metrics
# Evaluate all torchvision models
print("="*70)
print("FINAL EVALUATION ON TEST SET")
print("="*70)
all_results = []
for model_cfg in MODELS_TO_TRAIN:
    name = model_cfg["name"]
    weights_path = RUNS_DIR / name / "weights" / "best.pt"
    
    if not weights_path.exists():
        print(f"Weights not found for {name}, skipping...")
        continue
    
    metrics = evaluate_final(
        model_name=name,
        weights_path=weights_path,
        img_size=model_cfg["img_size"],
        device=device,
    )
    all_results.append(metrics)
    
    print(f"\n{name}:")
    print(f"  Parameters: {metrics['parameters_M']:.2f}M")
    print(f"  mAP50: {metrics['mAP50']:.4f}")
    print(f"  mAP50-95: {metrics['mAP50-95']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    if metrics['iou_stats']:
        print(f"  Mean IoU: {metrics['iou_stats']['mean_iou']:.4f}")
    print(f"  FPS: {metrics['speed_stats']['fps']:.1f}")

#### Load YOLO Results for Comparison

In [ ]:
# =============================================================================
# Cell 11: Load YOLO Results for Comparison
# =============================================================================
"""
Load the best YOLO model results from the previous notebook.
"""
# Path to YOLO results
YOLO_RUNS_DIR = PROJ_ROOT / "runs"
# Find YOLO models
yolo_models = []
for run_dir in YOLO_RUNS_DIR.iterdir():
    if run_dir.is_dir() and (run_dir / "weights" / "best.pt").exists():
        yolo_models.append(run_dir.name)
print("Found YOLO models:", yolo_models)
# Load YOLO metrics (we'll use the best one: yolov8n_mixed_lowlr)
yolo_results = []
for yolo_name in yolo_models:
    results_csv = YOLO_RUNS_DIR / yolo_name / "results.csv"
    
    if not results_csv.exists():
        continue
    
    df = pd.read_csv(results_csv)
    
    # Get best metrics
    best_idx = df["metrics/mAP50(B)"].idxmax()
    
    yolo_result = {
        "model_name": f"YOLOv8n ({yolo_name})",
        "parameters_M": 3.2,  # YOLOv8n has ~3.2M params
        "mAP50": df.loc[best_idx, "metrics/mAP50(B)"],
        "mAP50-95": df.loc[best_idx, "metrics/mAP50-95(B)"],
        "precision": df.loc[best_idx, "metrics/precision(B)"],
        "recall": df.loc[best_idx, "metrics/recall(B)"],
        "iou_stats": {"mean_iou": None},  # Need to compute separately
        "speed_stats": {"fps": None, "mean_inference_ms": None},
    }
    yolo_results.append(yolo_result)
    
    print(f"\n{yolo_result['model_name']}:")
    print(f"  mAP50: {yolo_result['mAP50']:.4f}")
    print(f"  mAP50-95: {yolo_result['mAP50-95']:.4f}")
# Combine all results
all_results.extend(yolo_results)

#### Comparison Table

In [ ]:
# =============================================================================
# Cell 12: Generate Comparison Table
# =============================================================================
"""
Create comprehensive comparison table of all models.
"""
comparison_data = []
for result in all_results:
    row = {
        "Model": result["model_name"],
        "Parameters (M)": result["parameters_M"],
        "mAP50": result["mAP50"],
        "mAP50-95": result["mAP50-95"],
        "Precision": result["precision"],
        "Recall": result["recall"],
        "Mean IoU": result["iou_stats"].get("mean_iou") if result["iou_stats"] else None,
        "FPS": result["speed_stats"].get("fps") if result["speed_stats"] else None,
        "Inference (ms)": result["speed_stats"].get("mean_inference_ms") if result["speed_stats"] else None,
    }
    comparison_data.append(row)
df_comparison = pd.DataFrame(comparison_data)
# Sort by mAP50-95
df_comparison = df_comparison.sort_values("mAP50-95", ascending=False)
print("="*70)
print("MODEL COMPARISON TABLE")
print("="*70)
print(df_comparison.to_string(index=False))
# Save to CSV
comparison_path = RUNS_DIR / "model_comparison_all.csv"
df_comparison.to_csv(comparison_path, index=False)
print(f"\nSaved to: {comparison_path}")

#### Visualization - Training Curves

In [ ]:
# =============================================================================
# Cell 13: Visualization - Training Curves
# =============================================================================
"""
Plot training curves for SSD and Faster R-CNN models.
"""
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model_cfg in MODELS_TO_TRAIN:
    name = model_cfg["name"]
    history_path = RUNS_DIR / name / "history.csv"
    
    if not history_path.exists():
        continue
    
    df = pd.read_csv(history_path)
    epochs = range(1, len(df) + 1)
    
    # Loss curve
    axes[0].plot(epochs, df["train_loss"], label=name, linewidth=2)
    
    # mAP curve
    axes[1].plot(epochs, df["val_mAP50"], label=f"{name} (mAP50)", linewidth=2)
    axes[1].plot(epochs, df["val_mAP50-95"], label=f"{name} (mAP50-95)", 
                 linewidth=2, linestyle="--")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("mAP")
axes[1].set_title("Validation mAP")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RUNS_DIR / "training_curves.png", dpi=150)
plt.show()
print(f"Saved: {RUNS_DIR / 'training_curves.png'}")

#### Visualization - Model Comparison Bar Chart

In [ ]:
# =============================================================================
# Cell 14: Visualization - Model Comparison Bar Chart
# =============================================================================
"""
Create bar chart comparing all models.
"""
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models = df_comparison["Model"].tolist()
x = np.arange(len(models))
width = 0.6
# mAP comparison
ax1 = axes[0]
bars1 = ax1.bar(x, df_comparison["mAP50"], width, label="mAP50", color="steelblue")
ax1.bar(x, df_comparison["mAP50-95"], width*0.6, label="mAP50-95", color="darkorange")
ax1.set_ylabel("mAP")
ax1.set_title("Detection Accuracy (mAP)")
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=45, ha="right")
ax1.legend()
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3, axis="y")
# Precision/Recall comparison
ax2 = axes[1]
width2 = 0.35
ax2.bar(x - width2/2, df_comparison["Precision"], width2, label="Precision", color="green")
ax2.bar(x + width2/2, df_comparison["Recall"], width2, label="Recall", color="red")
ax2.set_ylabel("Score")
ax2.set_title("Precision vs Recall")
ax2.set_xticks(x)
ax2.set_xticklabels(models, rotation=45, ha="right")
ax2.legend()
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3, axis="y")
# Parameters comparison
ax3 = axes[2]
ax3.bar(x, df_comparison["Parameters (M)"], width, color="purple")
ax3.set_ylabel("Parameters (M)")
ax3.set_title("Model Size")
ax3.set_xticks(x)
ax3.set_xticklabels(models, rotation=45, ha="right")
ax3.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(RUNS_DIR / "model_comparison.png", dpi=150)
plt.show()
print(f"Saved: {RUNS_DIR / 'model_comparison.png'}")

#### Sample Predictions Visualization

In [ ]:
# =============================================================================
# Cell 15: Sample Predictions Visualization
# =============================================================================
"""
Visualize predictions from all models on the same test images.
"""
def visualize_predictions_comparison(models_info, test_images, device):
    """
    Show predictions from multiple models side by side.
    models_info: list of (name, model, img_size)
    """
    n_models = len(models_info)
    n_images = len(test_images)
    
    fig, axes = plt.subplots(n_images, n_models, figsize=(5*n_models, 5*n_images))
    if n_images == 1:
        axes = [axes]
    
    for img_idx, img_path in enumerate(test_images):
        for model_idx, (name, model, img_size) in enumerate(models_info):
            ax = axes[img_idx][model_idx] if n_images > 1 else axes[model_idx]
            
            # Load and preprocess image
            img = Image.open(img_path).convert("RGB")
            img_resized = img.resize((img_size, img_size))
            img_tensor = transforms.ToTensor()(img_resized).unsqueeze(0).to(device)
            
            # Predict
            model.eval()
            with torch.no_grad():
                outputs = model(img_tensor)
            
            # Draw predictions
            img_np = np.array(img_resized)
            
            boxes = outputs[0]["boxes"].cpu().numpy()
            scores = outputs[0]["scores"].cpu().numpy()
            labels = outputs[0]["labels"].cpu().numpy()
            
            # Filter by confidence
            mask = (scores >= 0.5) & (labels == 1)
            boxes = boxes[mask]
            scores = scores[mask]
            
            for box, score in zip(boxes, scores):
                x1, y1, x2, y2 = box.astype(int)
                cv2.rectangle(img_np, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img_np, f"{score:.2f}", (x1, y1-5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            
            ax.imshow(img_np)
            ax.set_title(f"{name}\n{len(boxes)} detections")
            ax.axis("off")
    
    plt.tight_layout()
    return fig
# Load models for visualization
models_for_viz = []
for model_cfg in MODELS_TO_TRAIN:
    name = model_cfg["name"]
    weights_path = RUNS_DIR / name / "weights" / "best.pt"
    
    if not weights_path.exists():
        continue
    
    if "ssd" in name:
        model = create_ssdlite_model(num_classes=2)
    else:
        model = create_fasterrcnn_model(num_classes=2)
    
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()
    
    models_for_viz.append((name, model, model_cfg["img_size"]))
# Get sample test images
test_img_dir = YOLO_DATASET_DIR / "images" / "test"
test_images = list(test_img_dir.glob("*.jpg"))[:4]  # 4 sample images
if models_for_viz and test_images:
    fig = visualize_predictions_comparison(models_for_viz, test_images, device)
    plt.savefig(RUNS_DIR / "predictions_comparison.png", dpi=150)
    plt.show()
    print(f"Saved: {RUNS_DIR / 'predictions_comparison.png'}")
else:
    print("No models or images available for visualization")

#### Final Summary

In [ ]:
# =============================================================================
# Cell 16: Final Summary
# =============================================================================
"""
Print final summary and conclusions.
"""
print("="*70)
print("FINAL SUMMARY - HAND DETECTION MODEL COMPARISON")
print("="*70)
print("\n1. MODELS TRAINED:")
print("-"*50)
for model_cfg in MODELS_TO_TRAIN:
    name = model_cfg["name"]
    weights_path = RUNS_DIR / name / "weights" / "best.pt"
    status = "TRAINED" if weights_path.exists() else "NOT FOUND"
    print(f"  - {name}: {status}")
print("\n2. COMPARISON TABLE:")
print("-"*50)
print(df_comparison.to_string(index=False))
print("\n3. KEY FINDINGS:")
print("-"*50)
# Find best model for each metric
best_map50 = df_comparison.loc[df_comparison["mAP50"].idxmax()]
best_map5095 = df_comparison.loc[df_comparison["mAP50-95"].idxmax()]
smallest = df_comparison.loc[df_comparison["Parameters (M)"].idxmin()]
print(f"  Best mAP50: {best_map50['Model']} ({best_map50['mAP50']:.4f})")
print(f"  Best mAP50-95: {best_map5095['Model']} ({best_map5095['mAP50-95']:.4f})")
print(f"  Smallest model: {smallest['Model']} ({smallest['Parameters (M)']:.2f}M)")
print("\n4. OUTPUT FILES:")
print("-"*50)
print(f"  Comparison CSV: {RUNS_DIR / 'model_comparison_all.csv'}")
print(f"  Training curves: {RUNS_DIR / 'training_curves.png'}")
print(f"  Model comparison: {RUNS_DIR / 'model_comparison.png'}")
print(f"  Predictions: {RUNS_DIR / 'predictions_comparison.png'}")
print("\n5. NEXT STEPS:")
print("-"*50)
print("  1. Select the best model for the sign language pipeline")
print("  2. Proceed to letter classification training")
print("  3. Implement temporal modeling (LSTM/Transformer)")
print("  4. Build end-to-end video pipeline")
print("\n" + "="*70)
print("NOTEBOOK COMPLETE")
print("="*70)